
# Machine State Layer — Tests and Validation

This notebook tests the first implementation of the ISIS RCS `machine_state` layer.

It assumes the following files are in the same directory as this notebook:

- `cycle_time.py`
- `machine_state_defaults.py`
- `tune_control.py`
- `correctors.py`
- `machine_state.py`
- `machine_state_writer.py`

The notebook first tests the pure-Python machine-state layer, then runs a real cpymad/MAD-X tune-matching test for the `madx_match` path:

- beam-state input from `cycle_time.py`
- default magnet scaling presets
- tune-control conversion functions
- real cpymad/MAD-X tune matching via `match_tune_madx()` and `calculate_kqtf_kqtd_madx_match()`
- corrector current/kick conversions
- `MachineState` construction and updates
- writing timestamped MAD-X-readable machine-state files


# Machine State Layer — User Manual

This notebook tests and demonstrates the **machine state layer** of the ISIS RCS optics GUI backend.

The machine state layer is the second layer of the optics GUI model stack. The first layer, `cycle_time.py`, converts a selected cycle time into a beam state. This second layer takes that beam state and gathers all machine settings needed to describe the model at that time.

The aim is to create a single, explicit, reproducible object representing the ISIS RCS machine state at one point in the acceleration cycle.

---

## 1. Purpose of the machine state layer

The machine state layer answers the question:

> Given a selected cycle time, requested tune settings, magnet scaling mode, harmonic tune settings, corrector settings and optional error table, what complete set of MAD-X override variables should be applied to the lattice?

It is designed to:

* store the complete model state at one cycle time;
* use the beam parameters from `cycle_time.py`;
* keep the MAD-X lattice files unchanged;
* collect all GUI/backend settings in one Python object;
* write a timestamped MAD-X-readable override file;
* write a JSON sidecar file for logging, debugging and reproducibility;
* retain both corrector kicks and corrector currents, even though only kicks are currently written to MAD-X.

Most of this layer is pure Python. The exception is the optional `"madx_match"` tune method, which requires a live cpymad/MAD-X instance with the lattice already loaded and uses MAD-X matching to calculate `kqtf` and `kqtd`.

---

## 2. Files in this layer

The layer consists of five main files:

```text
machine_state_defaults.py
tune_control.py
correctors.py
machine_state.py
machine_state_writer.py
```

The notebook also uses:

```text
cycle_time.py
```

from the previous layer.

---

## 3. Layer inputs and outputs

The main input is a beam state from `cycle_time.py`:

```python
from cycle_time import RCSRamp

ramp = RCSRamp(top_energy_MeV=800.0)
beam_state = ramp.state_at(3.0)
```

The returned `beam_state` contains quantities such as:

```text
cycle_time_ms
kinetic_energy_MeV
total_energy_MeV
gamma
beta
momentum_MeV_c
brho_Tm
normalised_momentum
```

The machine state layer then combines this with:

```text
main magnet scaling mode
requested horizontal tune
requested vertical tune
base horizontal tune
base vertical tune
tune-control method
harmonic tune variables
HD/VD corrector kicks
HD/VD corrector currents
optional MAD-X error table path
metadata
```

The main output is a `MachineState` object, which can be written to a MAD-X-readable `.strength` file and a JSON summary file.

---

## 4. `machine_state_defaults.py`

`machine_state_defaults.py` stores the standard values used throughout the layer.

It contains the main magnet scaling presets:

```python
RCS_BARE_SCALING
SRM_BARE_SCALING
MAIN_MAGNET_SCALING_PRESETS
```

The two currently supported magnet modes are:

```text
"rcs_bare"
"srm_bare"
```

The file also stores the default tune settings:

```python
DEFAULT_QX = 4.331
DEFAULT_QY = 3.731

DEFAULT_BASE_QX = 4.331
DEFAULT_BASE_QY = 3.731

DEFAULT_TUNE_METHOD = "di_wright"
```

The default harmonic tune variables are:

```python
D7SIN
D7COS
D8SIN
D8COS
F8SIN
F8COS
F9SIN
F9COS
```

All harmonic variables default to zero.

The corrector names are separated into horizontal and vertical correctors:

```python
HD_CORRECTOR_NAMES
VD_CORRECTOR_NAMES
```

The operational corrector superperiods are:

```python
[0, 2, 3, 4, 5, 7, 9]
```

The corrector calibration dictionary converts controls-style currents in amperes into MAD-X kicks.

The conversion convention is:

$$
\theta_{\mathrm{mrad}} = \frac{I_{\mathrm{A}} C}{B\rho}
$$

where $I_{\mathrm{A}}$ is the corrector current in amperes, $C$ is the corrector calibration value and $B\rho$ is the magnetic rigidity in Tm.

The MAD-X kick is then stored in radians:

$$
\theta_{\mathrm{rad}} = 10^{-3}\theta_{\mathrm{mrad}}
$$

---

## 5. `tune_control.py`

`tune_control.py` handles the conversion from requested tunes to trim-quadrupole settings.

The currently implemented tune-control methods are:

```text
"di_wright"
"madx_match"
```

The main functions are:

```python
tune_to_trim_quad_current_di()
trim_quad_current_to_tune_di()
current_to_strength()
strength_to_current()
calculate_kqtf_kqtd_di()
match_tune_madx()
calculate_kqtf_kqtd_madx_match()
```

The Di Wright tune-control equations first calculate the QTF and QTD currents required to move from the base tunes to the requested tunes.

The current-to-strength conversion uses:

$$
k = \frac{I G_{\mathrm{cal}}}{B\rho p_n}
$$

where:

* $k$ is the MAD-X trim-quadrupole strength;
* $I$ is the trim-quadrupole current;
* $G_{\mathrm{cal}}$ is the trim-quadrupole calibration;
* $B\rho$ is the beam rigidity;
* $p_n$ is the normalised momentum.

The high-level helper:

```python
calculate_kqtf_kqtd_di()
```

returns:

```python
{
    "iqtf_A": ...,
    "iqtd_A": ...,
    "kqtf": ...,
    "kqtd": ...,
}
```

The `MachineState` object stores both the currents and strengths:

```text
iqtf_A
iqtd_A
kqtf
kqtd
```

The `"madx_match"` tune method uses a real MAD-X match through cpymad. It varies `kqtd` and `kqtf`, constrains `q1` and `q2`, optionally constrains `dq1` and `dq2`, runs the MAD-X `jacobian` matcher, then reads back the matched strengths.

A minimal direct use is:

```python
result = calculate_kqtf_kqtd_madx_match(
    qx=4.31,
    qy=3.76,
    beam_state=beam_state,
    madx_instance=madx,
    sequence_name="synchrotron",
)
```

The same path can be used through `MachineState`:

```python
state = MachineState.from_defaults(
    beam_state=beam_state,
    requested_qx=4.31,
    requested_qy=3.76,
    tune_method="madx_match",
    calculate_tune=False,
)

state.calculate_trim_quad_strengths(
    madx_instance=madx,
    sequence_name="synchrotron",
)
```

This requires the MAD-X lattice to have been loaded first, and the globals `kqtd` and `kqtf` must exist in the active MAD-X instance.

---

## 6. `correctors.py`

`correctors.py` manages the operational HD/VD corrector state.

The important design decision is that the machine state stores both:

```text
MAD-X kicks in radians
controls-style currents in amperes
```

Only the kicks are written to the MAD-X override file. The currents are retained for GUI display, logging and later correction-suggestion workflows.

The main helper functions are:

```python
default_hd_corrector_kicks_rad()
default_vd_corrector_kicks_rad()

default_hd_corrector_currents_A()
default_vd_corrector_currents_A()

current_to_kick_rad()
kick_rad_to_current()

currents_to_kicks_rad()
kicks_rad_to_currents_A()

build_corrector_state()
```

`build_corrector_state()` can treat either kicks or currents as the primary input.

For example, if `prefer="currents"`, the supplied currents are converted to MAD-X kicks:

```python
corrector_state = build_corrector_state(
    beam_state=beam_state,
    hd_corrector_currents_A=hd_currents,
    vd_corrector_currents_A=vd_currents,
    prefer="currents",
)
```

If `prefer="kicks"`, the supplied kicks are converted back into currents:

```python
corrector_state = build_corrector_state(
    beam_state=beam_state,
    hd_corrector_kicks_rad=hd_kicks,
    vd_corrector_kicks_rad=vd_kicks,
    prefer="kicks",
)
```

This allows the GUI to work naturally with controls-style current values while MAD-X receives the kick variables it needs.

---

## 7. `machine_state.py`

`machine_state.py` defines the central dataclass:

```python
MachineState
```

A `MachineState` represents the complete machine settings at one ISIS RCS cycle time.

It stores:

```text
timestamp
cycle_time_ms
beam_state

main_magnet_mode
main_magnet_scaling

tune_method
requested_qx
requested_qy
base_qx
base_qy

kqtd
kqtf
iqtd_A
iqtf_A

harmonic_tunes

error_table_path

hd_corrector_kicks_rad
vd_corrector_kicks_rad
hd_corrector_currents_A
vd_corrector_currents_A

metadata
```

The preferred constructor is:

```python
MachineState.from_defaults()
```

This starts from the standard defaults and applies only the requested overrides.

A minimal example is:

```python
from cycle_time import RCSRamp
from machine_state import MachineState

ramp = RCSRamp(top_energy_MeV=800.0)
beam_state = ramp.state_at(3.0)

machine_state = MachineState.from_defaults(
    beam_state=beam_state,
)
```

A more realistic example is:

```python
machine_state = MachineState.from_defaults(
    beam_state=beam_state,
    main_magnet_mode="srm_bare",
    requested_qx=4.316,
    requested_qy=3.765,
    tune_method="di_wright",
    harmonic_tunes={
        "D7SIN": 1.0,
        "D7COS": 0.0,
    },
    error_table_path="./error_tables/example_error_table.tfs",
    metadata={
        "label": "example state",
        "note": "test state for optics GUI workflow",
    },
)
```

During construction, the layer:

1. checks that the requested magnet mode is valid;
2. copies the corresponding magnet scaling preset;
3. completes the harmonic tune dictionary by filling unspecified terms with zero;
4. builds a consistent corrector state;
5. calculates `kqtf`, `kqtd`, `iqtf_A` and `iqtd_A` if `calculate_tune=True`.

The main update methods are:

```python
machine_state.calculate_trim_quad_strengths()
machine_state.update_harmonics()
machine_state.update_correctors()
machine_state.update_main_magnet_mode()
machine_state.update_error_table()
machine_state.summary_dict()
```

For example, harmonic settings can be updated using:

```python
machine_state.update_harmonics(
    D7SIN=2.0,
    D7COS=-1.0,
)
```

Correctors can be updated using currents:

```python
from collections import OrderedDict

hd_currents = machine_state.hd_corrector_currents_A.copy()
hd_currents["r3hd1_kick"] = 5.0

machine_state.update_correctors(
    hd_corrector_currents_A=hd_currents,
    prefer="currents",
)
```

or using kicks:

```python
hd_kicks = machine_state.hd_corrector_kicks_rad.copy()
hd_kicks["r3hd1_kick"] = 1.0e-4

machine_state.update_correctors(
    hd_corrector_kicks_rad=hd_kicks,
    prefer="kicks",
)
```

The complete machine state can be converted to a serialisable dictionary:

```python
summary = machine_state.summary_dict()
```

This is the object used by the JSON sidecar file.

---

## 8. `machine_state_writer.py`

`machine_state_writer.py` writes the machine state to disk.

The main function is:

```python
write_machine_state_file()
```

A typical use is:

```python
from machine_state_writer import write_machine_state_file

state_file = write_machine_state_file(
    machine_state,
    output_dir="./machine_states",
)
```

This creates a timestamped file with a name similar to:

```text
machine_state_20260514_153012_t3p000000ms.strength
```

The file is both:

```text
a MAD-X-readable override file
a human-readable machine-state log
```

By default, a JSON sidecar file is also written:

```text
machine_state_20260514_153012_t3p000000ms.json
```

The `.strength` file contains sections for:

```text
metadata comments
beam override
main magnet scaling
tune trim quadrupole settings
harmonic tune settings
HD corrector kicks
VD corrector kicks
HD corrector currents as comments
VD corrector currents as comments
JSON summary as comments
```

The MAD-X-readable assignments include entries such as:

```madx
brho := ...;
beam, particle=proton, pc=...;

qd_scale := ...;
qf_scale := ...;

kqtd := ...;
kqtf := ...;

D7SIN = ...;
D7COS = ...;

r0hd1_kick := ...;
r0vd1_kick := ...;
```

The corrector currents are written as comments because they are retained for logging and GUI display, but are not directly applied to MAD-X:

```madx
! r0hd1_kick_current_A = ...
! r0vd1_kick_current_A = ...
```

---

## 9. Standard workflow

The standard workflow is:

1. Build the beam state from a selected cycle time.
2. Build a `MachineState` from defaults.
3. Apply any requested tune, harmonic, corrector or error-table settings.
4. Write the state to a MAD-X-readable override file.
5. Later, the MAD-X model layer will load the base lattice and call this override file.

Example:

```python
from cycle_time import RCSRamp
from machine_state import MachineState
from machine_state_writer import write_machine_state_file

ramp = RCSRamp(top_energy_MeV=800.0)

beam_state = ramp.state_at(3.0)

machine_state = MachineState.from_defaults(
    beam_state=beam_state,
    main_magnet_mode="srm_bare",
    requested_qx=4.316,
    requested_qy=3.765,
    tune_method="di_wright",
    harmonic_tunes={
        "D7SIN": 0.0,
        "D7COS": 0.0,
        "D8SIN": 0.0,
        "D8COS": 0.0,
    },
    error_table_path=None,
    metadata={
        "case": "example default optics GUI state",
    },
)

state_file = write_machine_state_file(
    machine_state,
    output_dir="./machine_states",
)

print(state_file)
```

---

## 10. How this layer should be used by the later MAD-X model layer

The intended MAD-X/cpymad workflow is:

```text
load ISIS.injected_beam
load ISIS.strength
load ISIS.elements
load ISIS.sequence
call the generated machine-state override file
use the selected sequence
apply optional error table
run TWISS
```

The important point is that the generated machine-state file is an override file. It should not replace the base lattice files.

The base lattice remains the reference. The machine state file records the settings applied on top of it.

---

## 11. What this notebook tests

This notebook tests the Python layer only. It does not yet test a full MAD-X run.

The tests cover:

```text
imports and setup
beam-state input from cycle_time.py
default magnet scaling presets
default harmonic tune variables
HD/VD corrector naming
Di Wright tune-control functions
trim-quadrupole current/strength round trips
corrector current/kick round trips
MachineState construction
MachineState update methods
summary dictionary generation
machine-state file writing
JSON sidecar writing
expected failures for invalid inputs
```

The tests are intended to confirm that the state object is internally consistent before it is connected to the MAD-X execution layer.

---

## 12. Important conventions

### Cycle time

Cycle time is stored in milliseconds:

```text
cycle_time_ms
```

The beam state is generated externally by `RCSRamp.state_at()`.

### Magnet mode

The current recognised magnet modes are:

```text
"rcs_bare"
"srm_bare"
```

These select predefined main magnet scaling dictionaries.

### Tune method

The currently implemented tune method is:

```text
"di_wright"
```

The following modes are also recognised by the state object:

```text
"manual"
"none"
"madx_match"
```

`"manual"` and `"none"` leave `kqtf` and `kqtd` unchanged.

`"madx_match"` is reserved for the later MAD-X model layer and currently raises `NotImplementedError`.

### Corrector preference

The corrector state can be built from either kicks or currents:

```text
prefer="kicks"
prefer="currents"
```

Use `prefer="currents"` when starting from controls-system values.

Use `prefer="kicks"` when starting from MAD-X kick values.

### Error tables

The machine state stores the path to an error table:

```python
error_table_path
```

This layer does not apply the error table. It only records which table should be associated with this state. The later MAD-X model layer is responsible for applying it.

---

## 13. Design principle

The machine state layer is a snapshot layer.

It should make the model state explicit, reproducible and inspectable.

In practical terms, that means:

```text
one selected cycle time
one beam state
one set of magnet scaling values
one requested tune state
one harmonic tune state
one corrector state
one optional error table
one timestamped MAD-X override file
one JSON summary
```

This gives the GUI and backend a common object to pass around before MAD-X is run.



## 1. Imports and setup


In [1]:
from pathlib import Path
import os
import json
import math
import shutil
import tempfile
from collections import OrderedDict

import numpy as np
import pandas as pd

# Make notebook robust if run from another working directory.
# Change this path if the module files are stored elsewhere.
PROJECT_DIR = Path.cwd()

import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project directory:", PROJECT_DIR)


Project directory: /home/hr/Repositories/optics_gui/Dev/02_Machine_State


In [2]:

from cycle_time import RCSRamp, RCSState

import machine_state_defaults as defaults
from tune_control import (
    tune_to_trim_quad_current_di,
    trim_quad_current_to_tune_di,
    current_to_strength,
    strength_to_current,
    calculate_kqtf_kqtd_di,
)
from correctors import (
    default_hd_corrector_kicks_rad,
    default_vd_corrector_kicks_rad,
    default_hd_corrector_currents_A,
    default_vd_corrector_currents_A,
    default_corrector_kicks_rad,
    default_corrector_currents_A,
    corrector_calibration_key,
    current_to_kick_rad,
    kick_rad_to_current,
    currents_to_kicks_rad,
    kicks_rad_to_currents_A,
    split_hd_vd,
    build_corrector_state,
)
from machine_state import MachineState
from machine_state_writer import (
    make_machine_state_filename,
    format_madx_value,
    write_machine_state_file,
)

print("Imports successful")


Imports successful



## 2. Small test helpers


In [3]:

def assert_close(actual, expected, rtol=1e-12, atol=1e-15, message=""):
    if not np.isclose(actual, expected, rtol=rtol, atol=atol):
        raise AssertionError(
            message or f"Values differ: actual={actual}, expected={expected}"
        )


def assert_dict_keys_equal(dictionary, expected_keys, message=""):
    actual_keys = list(dictionary.keys())
    expected_keys = list(expected_keys)
    if actual_keys != expected_keys:
        raise AssertionError(
            message or f"Keys differ:\nactual={actual_keys}\nexpected={expected_keys}"
        )


def print_pass(name):
    print(f"PASS: {name}")



## 3. Beam-state object from `cycle_time.py`

The machine-state layer should consume `RCSState` rather than recalculating beam quantities internally.


In [4]:

ramp = RCSRamp(top_energy_MeV=800.0)
beam_state = ramp.state_at(3.0)

assert isinstance(beam_state, RCSState)
assert beam_state.cycle_time_ms == 3.0
assert beam_state.brho_Tm > 0.0
assert beam_state.momentum_GeV_c() > 0.0
assert beam_state.normalised_momentum > 1.0

print(beam_state)
print_pass("RCSState creation")


RCSState(t=3 ms, E_kin=172.565 MeV, p=0.594647 GeV/c, Brho=1.98353 Tm, pn=1.61093)
PASS: RCSState creation


## 3a. Cycle-time boundary assertions

These assertions lock down the current RCS ramp endpoints before the GUI layer starts relying on them.


In [5]:
injection_state = ramp.state_at(0.0)
extraction_state = ramp.state_at(10.0)
expected_p_ratio = ramp.top_momentum_MeV_c / ramp.injection_momentum_MeV_c

assert injection_state.cycle_time_ms == 0.0
assert extraction_state.cycle_time_ms == 10.0
assert_close(
    injection_state.kinetic_energy_MeV,
    ramp.injection_energy_MeV,
    rtol=1e-12,
    atol=1e-12,
)
assert_close(
    extraction_state.kinetic_energy_MeV,
    ramp.top_energy_MeV,
    rtol=1e-12,
    atol=1e-12,
)
assert_close(injection_state.normalised_momentum, 1.0, rtol=1e-12, atol=1e-12)
assert_close(
    extraction_state.normalised_momentum,
    expected_p_ratio,
    rtol=1e-12,
    atol=1e-12,
)

print_pass("cycle-time boundary states")


PASS: cycle-time boundary states


## 3b. Cycle-time out-of-range behaviour

The current implementation clamps beam quantities outside the modelled cycle range. The recorded `cycle_time_ms` remains the requested value.


In [6]:
below_range_state = ramp.state_at(-1.0)
above_range_state = ramp.state_at(11.0)

assert below_range_state.cycle_time_ms == -1.0
assert above_range_state.cycle_time_ms == 11.0
assert_close(
    below_range_state.kinetic_energy_MeV,
    injection_state.kinetic_energy_MeV,
    rtol=1e-12,
    atol=1e-12,
)
assert_close(
    above_range_state.kinetic_energy_MeV,
    extraction_state.kinetic_energy_MeV,
    rtol=1e-12,
    atol=1e-12,
)
assert_close(
    below_range_state.normalised_momentum,
    injection_state.normalised_momentum,
    rtol=1e-12,
    atol=1e-12,
)
assert_close(
    above_range_state.normalised_momentum,
    extraction_state.normalised_momentum,
    rtol=1e-12,
    atol=1e-12,
)

print_pass("cycle-time out-of-range values clamp beam quantities")


PASS: cycle-time out-of-range values clamp beam quantities



## 4. Defaults and naming conventions


In [7]:

expected_scaling_keys = [
    "main_dipole_scale",
    "fringe_dipole_scale",
    "fringe_dipole_neg_scale",
    "qd_scale",
    "qdfr_scale",
    "qf_scale",
    "qffr_scale",
    "qds_scale",
    "qdsfr_scale",
]

assert_dict_keys_equal(defaults.RCS_BARE_SCALING, expected_scaling_keys)
assert_dict_keys_equal(defaults.SRM_BARE_SCALING, expected_scaling_keys)

assert all(value == 1.0 for value in defaults.RCS_BARE_SCALING.values())
assert defaults.SRM_BARE_SCALING["qd_scale"] != 1.0
assert "rcs_bare" in defaults.MAIN_MAGNET_SCALING_PRESETS
assert "srm_bare" in defaults.MAIN_MAGNET_SCALING_PRESETS

print(defaults.RCS_BARE_SCALING)
print(defaults.SRM_BARE_SCALING)
print_pass("main magnet scaling defaults")


OrderedDict({'main_dipole_scale': 1.0, 'fringe_dipole_scale': 1.0, 'fringe_dipole_neg_scale': 1.0, 'qd_scale': 1.0, 'qdfr_scale': 1.0, 'qf_scale': 1.0, 'qffr_scale': 1.0, 'qds_scale': 1.0, 'qdsfr_scale': 1.0})
OrderedDict({'main_dipole_scale': 1.0, 'fringe_dipole_scale': 1.0, 'fringe_dipole_neg_scale': 1.0, 'qd_scale': 0.984962103600024, 'qdfr_scale': 0.994661334722062, 'qf_scale': 0.9887119657096543, 'qffr_scale': 0.9959758637608666, 'qds_scale': 0.9931335289706212, 'qdsfr_scale': 0.9958189246720367})
PASS: main magnet scaling defaults


In [8]:

expected_harmonics = [
    "D7SIN",
    "D7COS",
    "D8SIN",
    "D8COS",
    "F8SIN",
    "F8COS",
    "F9SIN",
    "F9COS",
]

assert_dict_keys_equal(defaults.DEFAULT_HARMONICS, expected_harmonics)
assert all(value == 0.0 for value in defaults.DEFAULT_HARMONICS.values())

print(defaults.DEFAULT_HARMONICS)
print_pass("harmonic defaults")


OrderedDict({'D7SIN': 0.0, 'D7COS': 0.0, 'D8SIN': 0.0, 'D8COS': 0.0, 'F8SIN': 0.0, 'F8COS': 0.0, 'F9SIN': 0.0, 'F9COS': 0.0})
PASS: harmonic defaults


In [9]:

expected_hd = [
    "r0hd1_kick",
    "r2hd1_kick",
    "r3hd1_kick",
    "r4hd1_kick",
    "r5hd1_kick",
    "r7hd1_kick",
    "r9hd1_kick",
]

expected_vd = [
    "r0vd1_kick",
    "r2vd1_kick",
    "r3vd1_kick",
    "r4vd1_kick",
    "r5vd1_kick",
    "r7vd1_kick",
    "r9vd1_kick",
]

assert defaults.HD_CORRECTOR_NAMES == expected_hd
assert defaults.VD_CORRECTOR_NAMES == expected_vd
assert_dict_keys_equal(defaults.DEFAULT_HD_CORRECTOR_KICKS_RAD, expected_hd)
assert_dict_keys_equal(defaults.DEFAULT_VD_CORRECTOR_KICKS_RAD, expected_vd)
assert_dict_keys_equal(defaults.DEFAULT_HD_CORRECTOR_CURRENTS_A, expected_hd)
assert_dict_keys_equal(defaults.DEFAULT_VD_CORRECTOR_CURRENTS_A, expected_vd)

print_pass("corrector names and defaults")


PASS: corrector names and defaults



## 5. Tune-control function tests


In [10]:

# At the base tune, the requested tune shift is zero, so the currents should be zero.
iqtf, iqtd = tune_to_trim_quad_current_di(
    qx=defaults.DEFAULT_BASE_QX,
    qy=defaults.DEFAULT_BASE_QY,
    base_qx=defaults.DEFAULT_BASE_QX,
    base_qy=defaults.DEFAULT_BASE_QY,
    pn=1.0,
)

assert_close(iqtf, 0.0)
assert_close(iqtd, 0.0)

print(iqtf, iqtd)
print_pass("Di tune current at base tune")


0.0 -0.0
PASS: Di tune current at base tune


In [11]:
# Round-trip: requested tune -> currents -> tune.
#
# The Di Wright helper returns the signed currents used by the MAD-X strength
# conversion. The inverse tune helper follows the controls-equation convention,
# so the returned currents are negated for this specific round-trip check.
requested_qx = 4.31
requested_qy = 3.76
pn = beam_state.normalised_momentum

iqtf, iqtd = tune_to_trim_quad_current_di(
    qx=requested_qx,
    qy=requested_qy,
    base_qx=defaults.DEFAULT_BASE_QX,
    base_qy=defaults.DEFAULT_BASE_QY,
    pn=pn,
)

qx_back, qy_back = trim_quad_current_to_tune_di(
    iqtf_A=-iqtf,
    iqtd_A=-iqtd,
    base_qx=defaults.DEFAULT_BASE_QX,
    base_qy=defaults.DEFAULT_BASE_QY,
    pn=pn,
)

assert_close(qx_back, requested_qx, rtol=1e-12, atol=1e-12)
assert_close(qy_back, requested_qy, rtol=1e-12, atol=1e-12)

print("requested:", requested_qx, requested_qy)
print("signed currents from tune_to_trim_quad_current_di:", iqtf, iqtd)
print("round trip:", qx_back, qy_back)
print_pass("Di tune current/tune round trip")

requested: 4.31 3.76
signed currents from tune_to_trim_quad_current_di: 4.602220279809223 -12.980324398520484
round trip: 4.31 3.76
PASS: Di tune current/tune round trip


In [12]:

# Current-strength conversion round trip.
current_A = 12.345
strength = current_to_strength(
    current_A=current_A,
    gcal=defaults.DEFAULT_TQ_GCAL,
    brho_Tm=beam_state.brho_Tm,
    pn=beam_state.normalised_momentum,
)
current_back = strength_to_current(
    strength=strength,
    gcal=defaults.DEFAULT_TQ_GCAL,
    brho_Tm=beam_state.brho_Tm,
    pn=beam_state.normalised_momentum,
)

assert_close(current_back, current_A)

print("current_A:", current_A)
print("strength:", strength)
print("current_back:", current_back)
print_pass("trim current/strength round trip")


current_A: 12.345
strength: 0.007715297500809936
current_back: 12.345000000000002
PASS: trim current/strength round trip


In [13]:

result = calculate_kqtf_kqtd_di(
    qx=requested_qx,
    qy=requested_qy,
    beam_state=beam_state,
)

expected_keys = {"iqtf_A", "iqtd_A", "kqtf", "kqtd"}
assert set(result.keys()) == expected_keys
assert isinstance(result["kqtf"], float)
assert isinstance(result["kqtd"], float)

result


{'iqtf_A': 4.602220279809223,
 'iqtd_A': -12.980324398520484,
 'kqtf': 0.0028762655830691696,
 'kqtd': -0.00811235839543194}


## 6. Corrector conversion tests


In [14]:

assert corrector_calibration_key("r0hd1_kick") == "0H"
assert corrector_calibration_key("r2vd1_kick") == "2V"
assert corrector_calibration_key("r9hd1_kick") == "9H"

print_pass("corrector calibration key parsing")


PASS: corrector calibration key parsing


In [15]:

# Single corrector current -> kick -> current round trip.
corrector_name = "r4hd1_kick"
current_A = 5.0

kick_rad = current_to_kick_rad(
    corrector_name=corrector_name,
    current_A=current_A,
    beam_state=beam_state,
)

current_back = kick_rad_to_current(
    corrector_name=corrector_name,
    kick_rad=kick_rad,
    beam_state=beam_state,
)

assert_close(current_back, current_A)

print("corrector:", corrector_name)
print("current_A:", current_A)
print("kick_rad:", kick_rad)
print("current_back:", current_back)
print_pass("single corrector current/kick round trip")


corrector: r4hd1_kick
current_A: 5.0
kick_rad: 0.00016637017529572034
current_back: 5.0
PASS: single corrector current/kick round trip


In [16]:

# Dictionary conversion round trip.
hd_currents = default_hd_corrector_currents_A()
hd_currents["r0hd1_kick"] = 1.0
hd_currents["r4hd1_kick"] = -2.0
hd_currents["r9hd1_kick"] = 3.5

hd_kicks = currents_to_kicks_rad(hd_currents, beam_state=beam_state)
hd_currents_back = kicks_rad_to_currents_A(hd_kicks, beam_state=beam_state)

for name in hd_currents:
    assert_close(hd_currents_back[name], hd_currents[name])

pd.DataFrame(
    {
        "current_A": hd_currents,
        "kick_rad": hd_kicks,
        "current_back_A": hd_currents_back,
    }
)


,current_A,kick_rad,current_back_A
r0hd1_kick,1.0,0.000042,1.0
r2hd1_kick,0.0,0.000000,0.0
r3hd1_kick,0.0,0.000000,0.0
r4hd1_kick,-2.0,-0.000067,-2.0
r5hd1_kick,0.0,0.000000,0.0
r7hd1_kick,0.0,0.000000,0.0
r9hd1_kick,3.5,0.000135,3.5


In [17]:

# Split HD/VD dictionary.
all_kicks = default_corrector_kicks_rad()
all_kicks["r0hd1_kick"] = 1e-6
all_kicks["r0vd1_kick"] = -2e-6

hd_split, vd_split = split_hd_vd(all_kicks)

assert "r0hd1_kick" in hd_split
assert "r0vd1_kick" in vd_split
assert "r0vd1_kick" not in hd_split
assert "r0hd1_kick" not in vd_split

print_pass("split HD/VD correctors")


PASS: split HD/VD correctors


In [18]:
# Build corrector state preferring currents.
hd_currents = default_hd_corrector_currents_A()
vd_currents = default_vd_corrector_currents_A()
hd_currents["r2hd1_kick"] = 2.0
vd_currents["r5vd1_kick"] = -3.0

state_from_currents = build_corrector_state(
    beam_state=beam_state,
    hd_corrector_currents_A=hd_currents,
    vd_corrector_currents_A=vd_currents,
    prefer="currents",
)

assert_close(state_from_currents["hd_corrector_currents_A"]["r2hd1_kick"], 2.0)
assert_close(state_from_currents["vd_corrector_currents_A"]["r5vd1_kick"], -3.0)
assert state_from_currents["hd_corrector_kicks_rad"]["r2hd1_kick"] != 0.0
assert state_from_currents["vd_corrector_kicks_rad"]["r5vd1_kick"] != 0.0

In [19]:
pd.DataFrame(
    {
        "hd_current_A": state_from_currents["hd_corrector_currents_A"],
        "hd_kick_rad": state_from_currents["hd_corrector_kicks_rad"],
    }
)

,hd_current_A,hd_kick_rad
r0hd1_kick,0.0,0.000000
r2hd1_kick,2.0,0.000092
r3hd1_kick,0.0,0.000000
r4hd1_kick,0.0,0.000000
r5hd1_kick,0.0,0.000000
r7hd1_kick,0.0,0.000000
r9hd1_kick,0.0,0.000000


In [20]:

# Build corrector state preferring kicks.
hd_kicks = default_hd_corrector_kicks_rad()
vd_kicks = default_vd_corrector_kicks_rad()
hd_kicks["r7hd1_kick"] = 1.2e-4
vd_kicks["r9vd1_kick"] = -0.8e-4

state_from_kicks = build_corrector_state(
    beam_state=beam_state,
    hd_corrector_kicks_rad=hd_kicks,
    vd_corrector_kicks_rad=vd_kicks,
    prefer="kicks",
)

assert_close(state_from_kicks["hd_corrector_kicks_rad"]["r7hd1_kick"], 1.2e-4)
assert_close(state_from_kicks["vd_corrector_kicks_rad"]["r9vd1_kick"], -0.8e-4)
assert state_from_kicks["hd_corrector_currents_A"]["r7hd1_kick"] != 0.0
assert state_from_kicks["vd_corrector_currents_A"]["r9vd1_kick"] != 0.0

pd.DataFrame(
    {
        "vd_kick_rad": state_from_kicks["vd_corrector_kicks_rad"],
        "vd_current_A": state_from_kicks["vd_corrector_currents_A"],
    }
)


,vd_kick_rad,vd_current_A
r0vd1_kick,0.00000,0.000000
r2vd1_kick,0.00000,0.000000
r3vd1_kick,0.00000,0.000000
r4vd1_kick,0.00000,0.000000
r5vd1_kick,0.00000,0.000000
r7vd1_kick,0.00000,0.000000
r9vd1_kick,-0.00008,-3.518454



## 7. `MachineState` construction tests


In [21]:

state_rcs = MachineState.from_defaults(
    beam_state=beam_state,
    main_magnet_mode="rcs_bare",
    requested_qx=defaults.DEFAULT_BASE_QX,
    requested_qy=defaults.DEFAULT_BASE_QY,
    tune_method="di_wright",
)

assert state_rcs.cycle_time_ms == beam_state.cycle_time_ms
assert state_rcs.main_magnet_mode == "rcs_bare"
assert all(value == 1.0 for value in state_rcs.main_magnet_scaling.values())
assert_close(state_rcs.kqtf, 0.0)
assert_close(state_rcs.kqtd, 0.0)
assert_close(state_rcs.iqtf_A, 0.0)
assert_close(state_rcs.iqtd_A, 0.0)

state_rcs.summary_dict().keys()


dict_keys(['timestamp', 'cycle_time_ms', 'beam_state', 'main_magnet_mode', 'main_magnet_scaling', 'tune_method', 'requested_qx', 'requested_qy', 'base_qx', 'base_qy', 'kqtd', 'kqtf', 'iqtd_A', 'iqtf_A', 'harmonic_tunes', 'error_table_path', 'hd_corrector_kicks_rad', 'vd_corrector_kicks_rad', 'hd_corrector_currents_A', 'vd_corrector_currents_A', 'metadata'])

In [22]:

state_srm = MachineState.from_defaults(
    beam_state=beam_state,
    main_magnet_mode="srm_bare",
    requested_qx=4.31,
    requested_qy=3.76,
    tune_method="di_wright",
)

assert state_srm.main_magnet_mode == "srm_bare"
assert state_srm.main_magnet_scaling["qd_scale"] == defaults.SRM_BARE_SCALING["qd_scale"]
assert isinstance(state_srm.kqtf, float)
assert isinstance(state_srm.kqtd, float)
assert state_srm.kqtf != 0.0
assert state_srm.kqtd != 0.0

print("kqtf:", state_srm.kqtf)
print("kqtd:", state_srm.kqtd)
print("iqtf_A:", state_srm.iqtf_A)
print("iqtd_A:", state_srm.iqtd_A)
print_pass("MachineState.from_defaults with SRM scaling")


kqtf: 0.0028762655830691696
kqtd: -0.00811235839543194
iqtf_A: 4.602220279809223
iqtd_A: -12.980324398520484
PASS: MachineState.from_defaults with SRM scaling


In [23]:

# Manual tune mode should preserve manually supplied kqtd/kqtf.
manual_state = MachineState.from_defaults(
    beam_state=beam_state,
    main_magnet_mode="rcs_bare",
    tune_method="manual",
    calculate_tune=False,
)

manual_state.kqtd = 1.23e-3
manual_state.kqtf = -4.56e-3
manual_state.calculate_trim_quad_strengths()

assert_close(manual_state.kqtd, 1.23e-3)
assert_close(manual_state.kqtf, -4.56e-3)

print_pass("manual tune mode preserves kqtd/kqtf")


PASS: manual tune mode preserves kqtd/kqtf


In [24]:

# Invalid main magnet mode should fail cleanly.
try:
    MachineState.from_defaults(
        beam_state=beam_state,
        main_magnet_mode="not_a_mode",
    )
    raise AssertionError("Expected ValueError for invalid main_magnet_mode")
except ValueError as exc:
    print("Caught expected error:", exc)

print_pass("invalid main magnet mode validation")


Caught expected error: Unknown main_magnet_mode 'not_a_mode'. Valid modes are ['rcs_bare', 'srm_bare'].
PASS: invalid main magnet mode validation



## 8. `MachineState` update-method tests


In [25]:

state = MachineState.from_defaults(
    beam_state=beam_state,
    main_magnet_mode="rcs_bare",
    requested_qx=4.31,
    requested_qy=3.76,
)

state.update_harmonics(D7SIN=1.0, D8COS=-2.5, F8SIN=0.25)

assert_close(state.harmonic_tunes["D7SIN"], 1.0)
assert_close(state.harmonic_tunes["D8COS"], -2.5)
assert_close(state.harmonic_tunes["F8SIN"], 0.25)

print(state.harmonic_tunes)
print_pass("harmonic update")


OrderedDict({'D7SIN': 1.0, 'D7COS': 0.0, 'D8SIN': 0.0, 'D8COS': -2.5, 'F8SIN': 0.25, 'F8COS': 0.0, 'F9SIN': 0.0, 'F9COS': 0.0})
PASS: harmonic update


In [26]:

try:
    state.update_harmonics(NOT_A_HARMONIC=1.0)
    raise AssertionError("Expected KeyError for invalid harmonic")
except KeyError as exc:
    print("Caught expected error:", exc)

print_pass("invalid harmonic validation")


Caught expected error: 'Unknown harmonic tune variable: NOT_A_HARMONIC'
PASS: invalid harmonic validation


In [27]:

state.update_main_magnet_mode("srm_bare")

assert state.main_magnet_mode == "srm_bare"
assert state.main_magnet_scaling["qf_scale"] == defaults.SRM_BARE_SCALING["qf_scale"]

state.update_main_magnet_mode("rcs_bare")
assert state.main_magnet_mode == "rcs_bare"
assert all(value == 1.0 for value in state.main_magnet_scaling.values())

print_pass("main magnet mode update")


PASS: main magnet mode update


In [28]:

hd_currents = default_hd_corrector_currents_A()
hd_currents["r3hd1_kick"] = 4.0

state.update_correctors(
    hd_corrector_currents_A=hd_currents,
    prefer="currents",
)

assert_close(state.hd_corrector_currents_A["r3hd1_kick"], 4.0)
assert state.hd_corrector_kicks_rad["r3hd1_kick"] != 0.0

print("r3hd1 current [A]:", state.hd_corrector_currents_A["r3hd1_kick"])
print("r3hd1 kick [rad]:", state.hd_corrector_kicks_rad["r3hd1_kick"])
print_pass("corrector update preferring currents")


r3hd1 current [A]: 4.0
r3hd1 kick [rad]: 0.00016132865483221365
PASS: corrector update preferring currents


In [29]:

state.update_error_table("survey_or_inferred_errors.tfs")
assert state.error_table_path == "survey_or_inferred_errors.tfs"

print_pass("error table path update")


PASS: error table path update



## 9. Summary dictionary tests


In [30]:

summary = state.summary_dict()

required_summary_keys = [
    "timestamp",
    "cycle_time_ms",
    "beam_state",
    "main_magnet_mode",
    "main_magnet_scaling",
    "tune_method",
    "requested_qx",
    "requested_qy",
    "base_qx",
    "base_qy",
    "kqtd",
    "kqtf",
    "iqtd_A",
    "iqtf_A",
    "harmonic_tunes",
    "error_table_path",
    "hd_corrector_kicks_rad",
    "vd_corrector_kicks_rad",
    "hd_corrector_currents_A",
    "vd_corrector_currents_A",
    "metadata",
]

for key in required_summary_keys:
    assert key in summary, f"Missing summary key: {key}"

assert "brho_Tm" in summary["beam_state"]
assert "momentum_GeV_c" in summary["beam_state"]

pd.Series(summary.keys())


0                   timestamp
1               cycle_time_ms
2                  beam_state
3            main_magnet_mode
4         main_magnet_scaling
5                 tune_method
6                requested_qx
7                requested_qy
8                     base_qx
9                     base_qy
10                       kqtd
11                       kqtf
12                     iqtd_A
13                     iqtf_A
14             harmonic_tunes
15           error_table_path
16     hd_corrector_kicks_rad
17     vd_corrector_kicks_rad
18    hd_corrector_currents_A
19    vd_corrector_currents_A
20                   metadata
dtype: object


## 10. Machine-state writer tests


In [31]:

assert format_madx_value(None) == "0.0"
assert format_madx_value(True) == "true"
assert format_madx_value(False) == "false"
assert format_madx_value(3) == "3"
assert isinstance(format_madx_value(3.14), str)

filename_preview = make_machine_state_filename(state, output_dir="./machine_states_test")
assert filename_preview.endswith(".strength")
assert "machine_state_" in filename_preview
assert "ms" in filename_preview

print(filename_preview)
print_pass("filename and MAD-X value formatting")


./machine_states_test/machine_state_20260610_214203_t3p000000ms.strength
PASS: filename and MAD-X value formatting


In [32]:

# Write one machine-state file and JSON sidecar into a temporary output folder.
test_output_dir = Path("./machine_states_test")
if test_output_dir.exists():
    shutil.rmtree(test_output_dir)

test_output_dir.mkdir(parents=True, exist_ok=True)

state_to_write = MachineState.from_defaults(
    beam_state=beam_state,
    main_magnet_mode="srm_bare",
    requested_qx=4.31,
    requested_qy=3.76,
    tune_method="di_wright",
    harmonic_tunes={"D7SIN": 1.0, "F8COS": -0.5},
    error_table_path="example_error_table.tfs",
)

hd_currents = default_hd_corrector_currents_A()
hd_currents["r0hd1_kick"] = 2.0
state_to_write.update_correctors(
    hd_corrector_currents_A=hd_currents,
    prefer="currents",
)

state_file = write_machine_state_file(
    state_to_write,
    output_dir=test_output_dir,
    write_json_sidecar=True,
)

state_file = Path(state_file)
json_file = state_file.with_suffix(".json")

assert state_file.exists()
assert json_file.exists()

print("Wrote:", state_file)
print("Wrote:", json_file)
print_pass("write machine-state file and JSON sidecar")


Wrote: machine_states_test/machine_state_20260610_214203_t3p000000ms.strength
Wrote: machine_states_test/machine_state_20260610_214203_t3p000000ms.json
PASS: write machine-state file and JSON sidecar


In [33]:

contents = state_file.read_text()

required_strings = [
    "ISIS RCS machine-state override file",
    "beam, particle=proton",
    "brho :=",
    "main_dipole_scale :=",
    "qd_scale :=",
    "kqtd :=",
    "kqtf :=",
    "D7SIN = 1",
    "F8COS = -0.5",
    "r0hd1_kick :=",
    "r0vd1_kick :=",
    "r0hd1_kick_current_A",
    "Machine-state JSON summary",
]

for text in required_strings:
    assert text in contents, f"Missing expected text: {text}"

print(contents[:2000])
print_pass("machine-state file content checks")


! ISIS RCS machine-state override file
! Generated by machine_state_writer.py
! timestamp = 2026-06-10 21:42:03
! cycle_time_ms = 3.0
! main_magnet_mode = srm_bare
! tune_method = di_wright
! requested_qx = 4.31
! requested_qy = 3.76
! base_qx = 4.331
! base_qy = 3.731
! error_table_path = example_error_table.tfs
!
! Beam state
! cycle_time_ms = 3.0
! kinetic_energy_MeV = 172.56519175739015
! kinetic_energy_GeV = 0.17256519175739016
! total_energy_MeV = 1110.83727991739
! total_energy_GeV = 1.1108372799173902
! gamma = 1.1839180701791945
! beta = 0.5353141487606986
! momentum_MeV_c = 594.6469129106272
! momentum_GeV_c = 0.5946469129106272
! brho_Tm = 1.983528594674077
! normalised_momentum = 1.6109349966163502
!
! Beam override
brho := 1.983528594674077;
beam, particle=proton, pc=0.5946469129106272;
!
! Main magnet scaling
main_dipole_scale := 1;
fringe_dipole_scale := 1;
fringe_dipole_neg_scale := 1;
qd_scale := 0.984962103600024;
qdfr_scale := 0.994661334722062;
qf_scale := 0.9887119

In [34]:

with open(json_file, "r") as f:
    json_summary = json.load(f)

assert json_summary["main_magnet_mode"] == "srm_bare"
assert json_summary["harmonic_tunes"]["D7SIN"] == 1.0
assert json_summary["harmonic_tunes"]["F8COS"] == -0.5
assert json_summary["error_table_path"] == "example_error_table.tfs"
assert json_summary["hd_corrector_currents_A"]["r0hd1_kick"] == 2.0

pd.Series(json_summary.keys())


0                   timestamp
1               cycle_time_ms
2                  beam_state
3            main_magnet_mode
4         main_magnet_scaling
5                 tune_method
6                requested_qx
7                requested_qy
8                     base_qx
9                     base_qy
10                       kqtd
11                       kqtf
12                     iqtd_A
13                     iqtf_A
14             harmonic_tunes
15           error_table_path
16     hd_corrector_kicks_rad
17     vd_corrector_kicks_rad
18    hd_corrector_currents_A
19    vd_corrector_currents_A
20                   metadata
dtype: object


## 11. End-to-end machine-state examples


In [35]:

def make_state_at_time(
    cycle_time_ms,
    main_magnet_mode="srm_bare",
    requested_qx=4.31,
    requested_qy=3.76,
):
    beam = ramp.state_at(cycle_time_ms)
    return MachineState.from_defaults(
        beam_state=beam,
        main_magnet_mode=main_magnet_mode,
        requested_qx=requested_qx,
        requested_qy=requested_qy,
        tune_method="di_wright",
    )

cycle_times = np.linspace(0.0, 10.0, 6)
states = [make_state_at_time(t) for t in cycle_times]

rows = []
for s in states:
    rows.append(
        {
            "cycle_time_ms": s.cycle_time_ms,
            "kinetic_energy_MeV": s.beam_state.kinetic_energy_MeV,
            "brho_Tm": s.beam_state.brho_Tm,
            "pn": s.beam_state.normalised_momentum,
            "kqtf": s.kqtf,
            "kqtd": s.kqtd,
            "iqtf_A": s.iqtf_A,
            "iqtd_A": s.iqtd_A,
        }
    )

df_states = pd.DataFrame(rows)
df_states


,cycle_time_ms,kinetic_energy_MeV,brho_Tm,pn,kqtf,kqtd,iqtf_A,iqtd_A
0,0.0,70.000000,1.231290,1.000000,0.004633,-0.013068,2.856863,-8.057634
1,2.0,112.758660,1.579809,1.283052,0.003611,-0.010185,3.665503,-10.338363
2,4.0,261.143070,2.492244,2.024092,0.002289,-0.006456,5.782552,-16.309390
3,6.0,496.359838,3.620076,2.940067,0.001576,-0.004445,8.399369,-23.689985
4,8.0,713.008113,4.532511,3.681107,0.001259,-0.003550,10.516417,-29.661011
5,10.0,800.000000,4.881030,3.964159,0.001169,-0.003297,11.325058,-31.941741


In [36]:

# Basic sanity checks for the time scan.
assert df_states["cycle_time_ms"].is_monotonic_increasing
assert df_states["kinetic_energy_MeV"].is_monotonic_increasing
assert df_states["brho_Tm"].is_monotonic_increasing
assert df_states["pn"].is_monotonic_increasing

print_pass("end-to-end time scan sanity checks")


PASS: end-to-end time scan sanity checks



## 12. Negative tests / expected failures

These tests confirm that invalid inputs fail cleanly.


In [37]:

# Invalid tune method.
try:
    bad_state = MachineState.from_defaults(
        beam_state=beam_state,
        tune_method="not_a_method",
        calculate_tune=False,
    )
    bad_state.calculate_trim_quad_strengths()
    raise AssertionError("Expected ValueError for invalid tune_method")
except ValueError as exc:
    print("Caught expected error:", exc)

print_pass("invalid tune method validation")


Caught expected error: Unknown tune_method: not_a_method
PASS: invalid tune method validation


In [38]:
# MAD-X tune matching boundary check.
#
# The machine-state layer stores the request only. The actual cpymad/MAD-X
# match is performed by MadxModel in Dev/03 and Dev/04.

madx_match_request = MachineState.from_defaults(
    beam_state=beam_state,
    requested_qx=4.31,
    requested_qy=3.76,
    tune_method="madx_match",
    calculate_tune=False,
)

madx_match_request.calculate_trim_quad_strengths()

assert madx_match_request.tune_method == "madx_match"
assert madx_match_request.requested_qx == 4.31
assert madx_match_request.requested_qy == 3.76
assert "madx_match" not in madx_match_request.metadata

print_pass("madx_match request remains a model-layer responsibility")


PASS: madx_match request remains a model-layer responsibility


In [39]:

# Invalid corrector name.
try:
    corrector_calibration_key("not_a_corrector")
    raise AssertionError("Expected ValueError for invalid corrector name")
except ValueError as exc:
    print("Caught expected error:", exc)

print_pass("invalid corrector name validation")


Caught expected error: Cannot parse corrector name: not_a_corrector
PASS: invalid corrector name validation


## 12a. Foundation failure-mode assertions

These checks make expected failures explicit for calibration and zero-rigidity inputs.


In [40]:
# Missing calibration for a recognised corrector should raise KeyError, not fall back silently.
try:
    current_to_kick_rad(
        corrector_name="r0hd1_kick",
        current_A=1.0,
        beam_state=beam_state,
        calibration={},
    )
    raise AssertionError("Expected KeyError for missing corrector calibration")
except KeyError as exc:
    print("Caught expected missing-calibration error:", exc)

# Zero Brho is non-physical for conversion and must fail clearly.
try:
    current_to_strength(
        current_A=1.0,
        gcal=defaults.DEFAULT_TQ_GCAL,
        brho_Tm=0.0,
        pn=beam_state.normalised_momentum,
    )
    raise AssertionError("Expected ZeroDivisionError for zero Brho")
except ZeroDivisionError as exc:
    print("Caught expected zero-Brho error:", exc)

print_pass("foundation failure modes")


Caught expected missing-calibration error: 'No corrector calibration found for r0hd1_kick (0H).'
Caught expected zero-Brho error: brho_Tm must be non-zero.
PASS: foundation failure modes



## 13. Final status


In [41]:

print("All machine-state layer tests completed successfully.")
print("Generated test output directory:", test_output_dir.resolve())
print("Generated files:")
for path in sorted(test_output_dir.glob("*")):
    print(" -", path.name)


All machine-state layer tests completed successfully.
Generated test output directory: /home/hr/Repositories/optics_gui/Dev/02_Machine_State/machine_states_test
Generated files:
 - machine_state_20260610_214203_t3p000000ms.json
 - machine_state_20260610_214203_t3p000000ms.strength
